# Generate per-class samples across DiffiT 256² training checkpoints

For each of the five checkpoints from `00017-diffit-256-gpus2-batch192`, generates `SAMPLES_PER_CLASS` images for each class and saves them to a single HDF5 file. The on-disk layout matches `san-v2/gen_images.py::RankH5Writer` (per-class groups, `images`/`seeds`/`written` datasets, lzf-compressed, chunked).

**Requested kimg → closest available snapshot:**

| Requested kimg | Snapshot file                 |
|---------------:|:------------------------------|
| 4500           | `network-snapshot-004435.pt` |
| 6300           | `network-snapshot-006451.pt` |
| 8000           | `network-snapshot-008064.pt` |
| 12000          | `network-snapshot-012096.pt` |
| 16000          | `network-snapshot-016128.pt` |

(Snapshots are spaced ~403 kimg apart, so exact matches don't exist.)

**Output:**

```
OUT_DIR/
  kimg_004435.h5
  kimg_006451.h5
  ...
```

Each `.h5` contains:
- file attrs: `format`, `image_shape_hwc`, `samples_per_class`, `classes`, `kimg`, `checkpoint`
- one group per class (`class_0`, `class_1`, …) with:
  - `images`  shape `(N, H, W, 3)`  uint8  chunked + lzf
  - `seeds`   shape `(N,)`           int64
  - `written` shape `(N,)`           bool   — used for resumption

**Compute scope:** With `SAMPLES_PER_CLASS=10000` and `num_classes=3`, this is 30K samples per checkpoint (150K total across the five). The loop reads `written` from disk on startup and skips any indices that are already done, so it's safely resumable.

## 1 — Imports and path setup

In [ ]:
import os, sys, time
from pathlib import Path

NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parent.parent          # .../DiffiT-v2
sys.path.insert(0, str(REPO_ROOT))

import h5py
import numpy as np
import torch
from tqdm.auto import tqdm
from diffusers.models import AutoencoderKL

import diffit.diffit as diffit_module
from diffit import create_diffusion, diffusion_defaults

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
print(f'h5py {h5py.__version__}')

## 2 — Configuration

All knobs in one place. The `CHECKPOINT_KIMGS` list is the *closest available* snapshot for each requested kimg — see the table at the top of this notebook.

In [ ]:
RUN_DIR = Path('/home/david/mnt/ssd_2_sata/python/phd/DiffiT-v2/training-runs/00017-diffit-256-gpus2-batch192')

# kimg → snapshot filename id (closest available to {4500, 6300, 8000, 12000, 16000})
CHECKPOINT_KIMGS = [4435, 6451, 8064, 12096, 16128]

SAMPLES_PER_CLASS  = 1000   # per (checkpoint, class)
BATCH_SIZE         = 32      # samples per forward pass per GPU; tune to GPU
NUM_SAMPLING_STEPS = 250
CFG_SCALE          = 4.4
USE_DDIM           = False
SCALE_POW          = 4.0     # cosine CFG schedule power for 256² (matches gen_images.py)
VAE_DECODER        = 'ema'   # 'ema' | 'mse'
BASE_SEED          = 0       # seed = BASE_SEED + kimg*10**6 + cls*10**3 + first_sample_idx

# HDF5 storage knobs (match san-v2 defaults)
HDF5_COMPRESSION   = 'lzf'   # 'lzf' | 'gzip' | None
HDF5_CHUNK_IMAGES  = 256     # chunk row count along the sample axis

OUT_DIR = REPO_ROOT / 'experiments' / 'samples' / '00017-diffit-256-gpus2-batch192'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Multi-GPU: one model + VAE copy per device, batches dispatched concurrently
# via a thread pool. Cap at 2 (or override with GPU_IDS = [0, 1, ...]).
GPU_IDS = list(range(min(2, torch.cuda.device_count()))) if torch.cuda.is_available() else []
DEVICES = [torch.device(f'cuda:{i}') for i in GPU_IDS] if GPU_IDS else [torch.device('cpu')]
torch.backends.cuda.matmul.allow_tf32 = True

# Verify all snapshots exist before starting a long run.
for k in CHECKPOINT_KIMGS:
    p = RUN_DIR / f'network-snapshot-{k:06d}.pt'
    assert p.exists(), f'Missing checkpoint: {p}'
    print(f'  ok  {p.name}')
print(f'\nOutput dir: {OUT_DIR}')
print(f'Devices: {DEVICES}')

## 3 — Load the VAE decoder (once)

The VAE is checkpoint-independent, so we keep it resident across all five model loads.

In [ ]:
# One VAE copy per device — small (~84M params) and avoids cross-GPU transfers
# when decoding each worker's latents.
VAES = {}
for dev in DEVICES:
    v = AutoencoderKL.from_pretrained(f'stabilityai/sd-vae-ft-{VAE_DECODER}').to(dev).eval()
    for p in v.parameters():
        p.requires_grad_(False)
    VAES[dev] = v

diff_config = diffusion_defaults()
diff_config['timestep_respacing'] = str(NUM_SAMPLING_STEPS)
diffusion = create_diffusion(**diff_config)
DIFFUSION_STEPS = diff_config['diffusion_steps']
print(f'VAE loaded on {len(VAES)} device(s); diffusion ready '
      f'(steps={NUM_SAMPLING_STEPS}, base_steps={DIFFUSION_STEPS})')

## 4 — Helpers

`load_diffit_model` follows the same EMA-loading logic as `experiments/analyze_sample_split.py::load_model` and infers `num_classes` from the embedding table.

`generate_class_batch` mirrors the CFG sampling block in `gen_images.py`.

In [ ]:
def load_diffit_model(ckpt_path, device, model_name='Diffit'):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if isinstance(ckpt, dict) and 'ema' in ckpt:
        state, image_size = ckpt['ema'], ckpt.get('image_size', 256)
    elif isinstance(ckpt, dict) and 'model' in ckpt:
        state, image_size = ckpt['model'], ckpt.get('image_size', 256)
    else:
        state, image_size = ckpt, 256

    # +1 slot is the CFG null token
    num_classes = state['y_embedder.embedding_table.weight'].shape[0] - 1
    latent_size = image_size // 8

    model = diffit_module.__dict__[model_name](
        input_size=latent_size, num_classes=num_classes,
    )
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f'  load_state_dict: missing={len(missing)} unexpected={len(unexpected)}')
    model.to(device).eval()
    return model, image_size, num_classes

In [ ]:
@torch.inference_mode()
def generate_class_batch(model, vae, device, class_idx, batch_size,
                         latent_size, num_classes, seed):
    """Generate a batch of `batch_size` images for a single class on `device`.
    Returns uint8 NHWC numpy."""
    gen = torch.Generator(device=device).manual_seed(int(seed))

    z = torch.randn(batch_size, 4, latent_size, latent_size,
                    device=device, generator=gen)
    classes      = torch.full((batch_size,), class_idx,    device=device, dtype=torch.long)
    classes_null = torch.full((batch_size,), num_classes,  device=device, dtype=torch.long)

    z_cfg = torch.cat([z, z], 0)
    model_kwargs = {
        'y':               torch.cat([classes, classes_null], 0),
        'cfg_scale':       CFG_SCALE,
        'diffusion_steps': DIFFUSION_STEPS,
        'scale_pow':       SCALE_POW,
    }

    sample_fn = diffusion.ddim_sample_loop if USE_DDIM else diffusion.p_sample_loop
    sample = sample_fn(
        model.forward_with_cfg,
        z_cfg.shape, z_cfg,
        clip_denoised=False, progress=False,
        model_kwargs=model_kwargs, device=device,
    )
    sample, _ = sample.chunk(2, dim=0)

    sample = vae.decode(sample / 0.18215).sample
    sample = ((sample + 1) * 127.5).clamp(0, 255).to(torch.uint8)
    return sample.permute(0, 2, 3, 1).contiguous().cpu().numpy()

In [ ]:
class H5Writer:
    """Single-file HDF5 writer for one checkpoint.

    Layout matches `san-v2/gen_images.py::RankH5Writer` (without sharding):

        file attrs: format, image_shape_hwc, samples_per_class, classes, kimg, checkpoint
        class_{c}/
          images   (N, H, W, C)  uint8   chunked, lzf
          seeds    (N,)          int64
          written  (N,)          bool
          attrs:   class_idx, samples_per_class, image_shape_hwc, written_count, missing_count

    Opens the file in append mode, so existing groups/datasets are reused on
    resumption. Writes are flushed after every batch.
    """
    def __init__(self, h5_path, classes, samples_per_class, image_shape_hwc,
                 compression='lzf', chunk_images=256, kimg=None, checkpoint=None):
        self.h5_path           = Path(h5_path)
        self.classes           = [int(c) for c in classes]
        self.samples_per_class = int(samples_per_class)
        self.image_shape_hwc   = tuple(int(x) for x in image_shape_hwc)
        self.compression       = None if (compression is None or str(compression).lower() == 'none') else str(compression).lower()
        self.chunk_images      = int(chunk_images)
        self.kimg              = kimg
        self.checkpoint        = checkpoint

        self.f = None
        self.d_images  = {}
        self.d_seeds   = {}
        self.d_written = {}

    def __enter__(self):
        self.h5_path.parent.mkdir(parents=True, exist_ok=True)
        self.f = h5py.File(str(self.h5_path), 'a')

        H, W, C = self.image_shape_hwc

        # Top-level attrs (overwrite each open — cheap and keeps them current).
        self.f.attrs['format']            = 'diffit_generated_images'
        self.f.attrs['image_shape_hwc']   = (H, W, C)
        self.f.attrs['samples_per_class'] = self.samples_per_class
        self.f.attrs['classes']           = np.array(self.classes, dtype=np.int32)
        if self.kimg       is not None: self.f.attrs['kimg']       = int(self.kimg)
        if self.checkpoint is not None: self.f.attrs['checkpoint'] = str(self.checkpoint)

        chunks0     = max(1, min(self.chunk_images,    self.samples_per_class))
        chunks_meta = max(1, min(self.chunk_images * 4, self.samples_per_class))

        for c in self.classes:
            gname = f'class_{c}'
            if gname in self.f:
                g = self.f[gname]
            else:
                g = self.f.create_group(gname)
                g.attrs['class_idx']         = int(c)
                g.attrs['samples_per_class'] = self.samples_per_class
                g.attrs['image_shape_hwc']   = (H, W, C)
                g.create_dataset(
                    'images',
                    shape=(self.samples_per_class, H, W, C),
                    dtype=np.uint8,
                    chunks=(chunks0, H, W, C),
                    compression=self.compression,
                    shuffle=bool(self.compression),
                )
                g.create_dataset('seeds',   shape=(self.samples_per_class,),
                                 dtype=np.int64, chunks=(chunks_meta,))
                g.create_dataset('written', shape=(self.samples_per_class,),
                                 dtype=np.bool_, chunks=(chunks_meta,))
                g['written'][:] = False

            self.d_images[c]  = g['images']
            self.d_seeds[c]   = g['seeds']
            self.d_written[c] = g['written']

        self.f.flush()
        return self

    def written_mask(self, class_idx):
        """Bool array of shape (samples_per_class,) — which sample indices already exist on disk."""
        return np.asarray(self.d_written[int(class_idx)][:], dtype=bool)

    def write_batch(self, class_idx, sample_idxs, seeds, images):
        sample_idxs = np.asarray(sample_idxs, dtype=np.int64)
        seeds       = np.asarray(seeds,       dtype=np.int64)
        images      = np.asarray(images,      dtype=np.uint8)

        if tuple(images.shape[1:]) != tuple(self.image_shape_hwc):
            raise RuntimeError(f'Shape mismatch: got {images.shape[1:]}, expected {self.image_shape_hwc}')
        if sample_idxs.size != images.shape[0] or seeds.size != images.shape[0]:
            raise RuntimeError('idxs/seeds/images batch size mismatch')

        # h5py requires monotonic indices for fancy assignment.
        order = np.argsort(sample_idxs)
        sample_idxs = sample_idxs[order]
        seeds       = seeds[order]
        images      = images[order]

        c = int(class_idx)
        self.d_images[c][sample_idxs, :, :, :] = images
        self.d_seeds[c][sample_idxs]           = seeds
        self.d_written[c][sample_idxs]         = True
        self.f.flush()

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.f is None:
            return
        for c in self.classes:
            written = int(np.count_nonzero(self.d_written[c][:]))
            g = self.f[f'class_{c}']
            g.attrs['written_count'] = written
            g.attrs['missing_count'] = self.samples_per_class - written
        self.f.flush()
        self.f.close()
        self.f = None

## 5 — Generation loop

Output layout:

```
OUT_DIR/
  kimg_004435.h5
  kimg_006451.h5
  kimg_008064.h5
  kimg_012096.h5
  kimg_016128.h5
```

Each file holds one group per class: `class_0`, `class_1`, …, with `images` / `seeds` / `written` datasets. The loop reads `written` from disk for each class on startup and only generates the indices that are still `False`, so killing the kernel mid-run and restarting will pick up where it left off.

Set `CLASS_FILTER` to a list of ints to generate only a subset of classes (useful for testing).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

CLASS_FILTER = None   # e.g. [0, 1, 2] for a smoke test, or None for all classes

for kimg in CHECKPOINT_KIMGS:
    ckpt_path = RUN_DIR / f'network-snapshot-{kimg:06d}.pt'
    h5_path   = OUT_DIR  / f'kimg_{kimg:06d}.h5'
    print(f'\n=== kimg={kimg}  ({ckpt_path.name}) -> {h5_path.name} ===')

    # Load one model copy per device.
    t_load = time.time()
    models = []
    image_size = num_classes = None
    for dev in DEVICES:
        m, image_size, num_classes = load_diffit_model(str(ckpt_path), dev)
        models.append(m)
    latent_size = image_size // 8
    print(f'  loaded on {len(DEVICES)} device(s) in {time.time() - t_load:.1f}s  '
          f'image_size={image_size}  num_classes={num_classes}  latent_size={latent_size}')

    classes = list(range(num_classes)) if CLASS_FILTER is None else list(CLASS_FILTER)

    with H5Writer(
        h5_path,
        classes=classes,
        samples_per_class=SAMPLES_PER_CLASS,
        image_shape_hwc=(image_size, image_size, 3),
        compression=HDF5_COMPRESSION,
        chunk_images=HDF5_CHUNK_IMAGES,
        kimg=kimg,
        checkpoint=ckpt_path.name,
    ) as writer:
        pending_per_class = {c: np.where(~writer.written_mask(c))[0] for c in classes}
        total_pending = sum(int(p.size) for p in pending_per_class.values())
        already_done  = len(classes) * SAMPLES_PER_CLASS - total_pending
        if already_done:
            print(f'  resuming: {already_done} samples already on disk')

        # Flatten work: one job per batch.
        jobs = []  # (cls, batch_ids, seed)
        for cls in classes:
            pending = pending_per_class[cls]
            for start in range(0, pending.size, BATCH_SIZE):
                batch_ids = pending[start:start + BATCH_SIZE]
                # Seed keyed off first sample idx — stable across resume.
                seed = BASE_SEED + kimg * 10**6 + cls * 10**3 + int(batch_ids[0])
                jobs.append((cls, batch_ids, seed))

        # Pre-shard jobs across GPUs so each worker thread is pinned to one
        # device — prevents concurrent access to the same model.
        jobs_per_gpu = [[] for _ in DEVICES]
        for i, job in enumerate(jobs):
            jobs_per_gpu[i % len(DEVICES)].append(job)

        write_lock = Lock()
        pbar = tqdm(total=total_pending, desc=f'kimg={kimg}', unit='img', smoothing=0.05)

        def gpu_worker(gpu_idx):
            dev   = DEVICES[gpu_idx]
            mdl   = models[gpu_idx]
            vae_d = VAES[dev]
            for cls, batch_ids, seed in jobs_per_gpu[gpu_idx]:
                bs = int(batch_ids.size)
                imgs = generate_class_batch(mdl, vae_d, dev, cls, bs,
                                            latent_size, num_classes, seed)
                seeds_arr = np.full(bs, seed, dtype=np.int64)
                with write_lock:
                    writer.write_batch(cls, batch_ids, seeds_arr, imgs)
                    pbar.update(bs)

        try:
            with ThreadPoolExecutor(max_workers=len(DEVICES)) as pool:
                futs = [pool.submit(gpu_worker, i) for i in range(len(DEVICES))]
                for f in futs:
                    f.result()  # surface exceptions
        finally:
            pbar.close()

    for m in models:
        del m
    models = []
    torch.cuda.empty_cache()
    print(f'  done kimg={kimg}  →  {h5_path}')

print('\nAll checkpoints done.')

## 6 — Quick sanity check (optional)

Load one batch from one class file and display a 4×4 grid to verify the saved arrays look like images of the expected class.

In [ ]:
import matplotlib.pyplot as plt

PEEK_KIMG  = CHECKPOINT_KIMGS[-1]   # most-trained checkpoint
PEEK_CLASS = 0                      # one of the classes

peek_path = OUT_DIR / f'kimg_{PEEK_KIMG:06d}.h5'
if not peek_path.exists():
    print(f'No file at {peek_path}; run cell 5 first.')
else:
    with h5py.File(str(peek_path), 'r') as f:
        print(f'{peek_path.name}  attrs:')
        for k, v in f.attrs.items():
            print(f'  {k}: {v}')

        gname = f'class_{PEEK_CLASS}'
        if gname not in f:
            print(f'\nNo group "{gname}" in file (classes present: {list(f.keys())})')
        else:
            g       = f[gname]
            written = np.asarray(g['written'][:], dtype=bool)
            n_done  = int(written.sum())
            print(f'\n{gname}: {n_done}/{int(g.attrs["samples_per_class"])} written  '
                  f'(images shape={g["images"].shape}, dtype={g["images"].dtype})')

            idxs = np.nonzero(written)[0][:16]
            if idxs.size == 0:
                print('  no images written yet for this class')
            else:
                imgs = g['images'][idxs]
                fig, axes = plt.subplots(4, 4, figsize=(10, 10))
                for ax, img in zip(axes.flat, imgs):
                    ax.imshow(img); ax.axis('off')
                # Pad unused subplots if fewer than 16 images.
                for ax in axes.flat[len(imgs):]:
                    ax.axis('off')
                fig.suptitle(f'kimg={PEEK_KIMG}  class={PEEK_CLASS}  '
                             f'(showing {len(imgs)} of {n_done})')
                fig.tight_layout()
                plt.show()